# Convulational Neural Network for CIFAR10 Dataset

## Overview

The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images.

Here are the classes in the dataset, as well as 10 random images from each:
- airplane
- automobile				
- bird										
- cat										
- deer										
- dog										
- frog										
- horse										
- ship										
- truck

The classes are completely mutually exclusive. There is no overlap between automobiles and trucks. "Automobile" includes sedans, SUVs, things of that sort. "Truck" includes only big trucks. Neither includes pickup trucks. We can use it to demonstrate how a CNN can learn to recognize objects in images:

1. Task & dataset
2. Data cleaning & preparation
3. Exploratory Data Analysis (EDA)
4. Train/validation/test split
5. Model training (forward pass, backprop, optimiser)
6. Evaluation metrics & loss curves
7. Error analysis
8. Pros, cons & real-world examples

## Setup and Imports

In [2]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.22.0-rc0


## 1. Load the CIFAR10 dataset

The dataset is divided into five training batches and one test batch, each with 10000 images. The test batch contains exactly 1000 randomly-selected images from each class. The training batches contain the remaining images in random order, but some training batches may contain more images from one class than another. Between them, the training batches contain exactly 5000 images from each class.

In [3]:
data_dir = Path("data")

def load_batch(filename):
    with open(data_dir / filename, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    images = batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    labels = np.asarray(batch[b"labels"])[:, None]
    return images, labels

train_batches = [load_batch(f"data_batch_{i}") for i in range(1, 6)]
x_train = np.concatenate([x for x, _ in train_batches])
y_train = np.concatenate([y for _, y in train_batches])
x_test, y_test = load_batch("test_batch")

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(50000, 32, 32, 3) (50000, 1) (10000, 32, 32, 3) (10000, 1)


In [4]:
class_names = [
    "airplane", "automobile", "bird", 
    "cat", "deer", "dog", "frog", "horse", "ship", "truck"
]
print(list(zip(np.unique(y_train), class_names)))

[(np.int64(0), 'airplane'), (np.int64(1), 'automobile'), (np.int64(2), 'bird'), (np.int64(3), 'cat'), (np.int64(4), 'deer'), (np.int64(5), 'dog'), (np.int64(6), 'frog'), (np.int64(7), 'horse'), (np.int64(8), 'ship'), (np.int64(9), 'truck')]
